In [ ]:
!pip install keras-tuner

In [ ]:
# import necessary stuff

import pandas as pd
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt
import PIL
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.layers import *
from keras.models import Model, load_model
from tensorflow.keras.initializers import glorot_uniform
from tensorflow.keras.utils import plot_model
from IPython.display import display
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint, LearningRateScheduler
import h5py as h5py
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dropout, Flatten, Dense
from tensorflow.keras import applications, optimizers
from tensorflow.keras.regularizers import l2
import pathlib
import keras
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import pathlib
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
train_data = "/content/drive/MyDrive/defense/split_dataset/train"
valid_data = "/content/drive/MyDrive/defense/split_dataset/val"
test_data = "/content/drive/MyDrive/defense/split_dataset/test"
dataset_dir1= pathlib.Path(train_data)
dataset_dir2= pathlib.Path(valid_data)
dataset_dir3= pathlib.Path(test_data)

In [ ]:
image_count = len(list(dataset_dir1.glob('*/*.jpg')))
print('Number of training image',image_count)
image_count2 = len(list(dataset_dir2.glob('*/*.jpg')))
print('Number of validation image',image_count2)
image_count3 = len(list(dataset_dir3.glob('*/*.jpg')))
print('Number of testing image',image_count3)

In [ ]:
img_height, img_width = 150, 150
epochs = 70
IMAGE_SIZE = [224, 224]

In [ ]:
print(os.listdir(train_data))

In [ ]:
print(os.listdir(valid_data))

In [ ]:
print(os.listdir(valid_data))

In [ ]:
image_count = len(list(dataset_dir1.glob('*/*.jpg')))
print('Number of training image',image_count)


image_count2 = len(list(dataset_dir2.glob('*/*.jpg')))
print('Number of validation image',image_count2)
image_count3 = len(list(dataset_dir3.glob('*/*.jpg')))
print('Number of testing image',image_count3)

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

valid_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    train_data,
    target_size=(224, 224),
    batch_size=64,
    class_mode='categorical'
)

valid_generator = valid_datagen.flow_from_directory(
    valid_data,
    target_size=(224, 224),
    batch_size=64,
    class_mode='categorical'
)


In [ ]:
input_shape = (224, 224, 3)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
# Load the ResNetv2 model (pre-trained on ImageNet)
base_model = tf.keras.applications.ResNet50V2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))


In [ ]:
input_tensor = Input(shape=(224, 224, 3))

In [ ]:
from tensorflow.keras import layers, models
# Freeze the layers of the pre-trained model
for layer in base_model.layers:
    layer.trainable = False

x = base_model(input_tensor)

# Add a global average pooling layer to reduce spatial dimensions
x = GlobalAveragePooling2D()(x)

# Add a fully connected layer for classification (adjust the number of units according to your needs)
x = Dense(256, activation='relu')(x)
x = Dense(128, activation='relu')(x)
x = Dense(64, activation='relu')(x)
# Add the final classification layer with the appropriate number of classes (e.g., 1000 for ImageNet)
output = Dense(12, activation='softmax')(x)

# Create the ResNet-50 V2 model
model = Model(inputs=input_tensor, outputs=output)

In [ ]:
model.summary()

In [ ]:
model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/defense/save/Finalversion.weights.h5",
    save_best_only=True,
    monitor='val_accuracy',
    save_freq='epoch',
    save_weights_only=True  # Ensure this is set to True
)


In [ ]:
model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

In [ ]:
import time
start_time = time.time()

In [ ]:
# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=70,
    validation_data=valid_generator,
    validation_steps=valid_generator.samples // valid_generator.batch_size,
    callbacks=[model_checkpoint]
)

In [ ]:
end_time = time.time()

In [ ]:
# Calculate and print the training duration
training_duration = end_time - start_time
print(f"Training completed in: {training_duration} seconds")

In [ ]:
import matplotlib.pyplot as plt

# Plot and save loss
plt.plot(model.history.history['loss'], label='train loss')
plt.plot(model.history.history['val_loss'], label='val loss')
plt.legend()
plt.savefig('resnet_LossVal_loss')
plt.show()

# Plot and save accuracies
plt.plot(model.history.history['accuracy'], label='train acc')
plt.plot(model.history.history['val_accuracy'], label='val acc')
plt.legend()
plt.savefig('resnet_AccVal_acc')
plt.show()

In [ ]:
train_loss, train_acc = model.evaluate(train_generator)
print("Train accuracy:", train_acc)
valid_loss, valid_acc = model.evaluate(valid_generator)
print("validation accuracy:", valid_acc)

In [ ]:
test_datagen = ImageDataGenerator(rescale=1.0 / 255)
test_generator = test_datagen.flow_from_directory(
    test_data,
    target_size=(224, 224),
    batch_size=64,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
test_loss, test_acc = model.evaluate(test_generator)
print("Test accuracy:", test_acc)

In [ ]:
tsdata = ImageDataGenerator(rescale=1/255)
test_datagen = tsdata.flow_from_directory(directory="/content/drive/MyDrive/defense/split_dataset/test", target_size=(224,224),shuffle=False, class_mode='categorical')

In [ ]:
test_datagen .class_indices

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Assuming you have true class labels in 'true_labels'
true_labels = test_datagen.classes

# Assuming 'pred_probs' contains your predicted probabilities with shape (number_of_samples, number_of_classes)
pred_probs = model.predict(test_datagen)

# Convert probabilities to class labels
predicted_labels = np.argmax(pred_probs, axis=1)

# Calculate confusion matrix
confusion = confusion_matrix(true_labels, predicted_labels)

# Get class labels from test_datagen.class_indices
class_labels = list(test_datagen.class_indices.keys())

# Create a colorful heatmap of the confusion matrix
plt.figure(figsize=(8, 6))
sns.set(font_scale=1.2)  # Adjust the font size for labels

# Create the heatmap with custom colors (you can choose your own colormap)
sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=class_labels, yticklabels=class_labels)

# Add labels and title
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix")

# Show the heatmap
plt.show()

# Generate classification report with precision, recall, F1-score, and class accuracy
report = classification_report(true_labels, predicted_labels, target_names=class_labels, output_dict=True)
accuracy = accuracy_score(true_labels, predicted_labels)

print("Classification Report:")
for class_label in class_labels:
    print(f"Class: {class_label}")
    print(f"  Precision: {report[class_label]['precision']:.2f}")
    print(f"  Recall: {report[class_label]['recall']:.2f}")
    print(f"  F1-score: {report[class_label]['f1-score']:.2f}")
    print(f"  Class Accuracy: {confusion[class_labels.index(class_label), class_labels.index(class_label)] / np.sum(confusion[class_labels.index(class_label), :]):.2%}")
    print(f"  Support: {report[class_label]['support']}")
    print()

# Overall metrics
print(f"Overall Precision: {report['macro avg']['precision']:.2f}")
print(f"Overall Recall: {report['macro avg']['recall']:.2f}")
print(f"Overall F1-score: {report['macro avg']['f1-score']:.2f}")
print(f"Overall Accuracy: {accuracy:.2%}")

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, jaccard_score
import numpy as np
from sklearn.metrics import log_loss, accuracy_score, cohen_kappa_score, matthews_corrcoef, roc_auc_score, roc_curve, auc, jaccard_score
from sklearn.model_selection import cross_val_score
# Assuming y_true and y_pred are your true labels and predicted labels, respectively
y_true = test_generator.classes
y_pred = model.predict(test_generator)
# Convert predicted probabilities to class labels
y_pred_labels = np.argmax(y_pred, axis=1)



# Calculate misclassification error
misclassification_error = 1 - accuracy_score(y_true, y_pred_labels)

# Calculate Jaccard similarity score
jaccard_similarity = jaccard_score(y_true, y_pred_labels, average='micro')

# Calculate Cohen's Kappa
kappa = cohen_kappa_score(y_true, y_pred_labels)

# Calculate Matthews Correlation Coefficient (MCC)
mcc = matthews_corrcoef(y_true, y_pred_labels)

print("Misclassification Error:", misclassification_error)
print("Jaccard Similarity Score:", jaccard_similarity)
print("Cohen's Kappa:", kappa)
print("Matthews Correlation Coefficient (MCC):", mcc)